# 08 - FAISS Indexing

## Objective

Create a FAISS vector index for Pipeline 2.

Pipeline:

Documents
→ Section-Based Chunking
→ BGE-M3 Embeddings
→ FAISS
→ Qwen3:14B

## Outcome

- Generate embeddings
- Create FAISS index
- Persist index
- Validate retrieval

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Load Oracle ERP Documents

Load and validate source documents.

In [2]:
from src.config import DOCS_PATH
from src.loaders.document_loader import DocumentLoader

loader = DocumentLoader(DOCS_PATH)

documents = loader.load()

print(
    f"Source Pages: {len(documents):,}"
)

Starting Document Loading...

Loading PDF : iSupplier_portal.pdf

Loading PDF : Purchasing_manual.pdf


DOCUMENT LOADING SUMMARY
PDF Files Loaded   : 2
DOCX Files Loaded  : 0
XLSX Files Loaded  : 0
Total Documents    : 1502
Source Pages: 1,502


## Create Section-Based Chunks

Create larger context-preserving chunks for Pipeline 2.

In [3]:
from langchain_core.documents import Document

In [4]:
def create_section_chunks(
    documents,
    target_size=2000
):

    section_chunks = []

    current_text = ""

    current_metadata = None

    for doc in documents:

        page_text = doc.page_content.strip()

        if not page_text:
            continue

        if current_metadata is None:
            current_metadata = doc.metadata.copy()

        current_text += "\n\n" + page_text

        if len(current_text) >= target_size:

            section_chunks.append(
                Document(
                    page_content=current_text.strip(),
                    metadata=current_metadata
                )
            )

            current_text = ""
            current_metadata = None

    if current_text:

        section_chunks.append(
            Document(
                page_content=current_text.strip(),
                metadata=current_metadata
            )
        )

    return section_chunks

In [5]:
section_chunks = create_section_chunks(
    documents
)

print(
    f"Section Chunks Created: {len(section_chunks):,}"
)

Section Chunks Created: 880


## Detect Hardware

Identify whether embeddings will run on CPU or GPU.

In [6]:
import torch

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    f"Embedding Device: {device}"
)

if torch.cuda.is_available():

    print(
        f"GPU : {torch.cuda.get_device_name(0)}"
    )

Embedding Device: cuda
GPU : NVIDIA RTX A5000 Laptop GPU


## Load BGE-M3 Embedding Model

In [7]:
from langchain_huggingface import (
    HuggingFaceEmbeddings
)

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={
        "device": device
    }
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## Generate Chunk Embeddings

Create vector embeddings for all section chunks.

In [8]:
chunk_texts = [
    chunk.page_content
    for chunk in section_chunks
]

print("=" * 80)

print("CHUNK EXTRACTION")

print("=" * 80)

print(
    f"Section Chunks : {len(section_chunks):,}"
)

print(
    f"Texts To Embed : {len(chunk_texts):,}"
)

print("\nSample Chunk Preview:\n")

print(
    chunk_texts[0][:500]
)

CHUNK EXTRACTION
Section Chunks : 880
Texts To Embed : 880

Sample Chunk Preview:

Oracle® iSupplier Portal
 User's Guide 
Release 12.2
 Part No. E48972-13
October 2025

Oracle iSupplier Portal User's Guide , Release 12.2
Part No. E48972-13
Copyright © 2009, 2025, Oracle and/or its affiliates. 
Primary Author:     Chetna Arora
Contributing Author:     Gowri Arur, Pragya Nair, Smile Nagpal
This software and related documentation are provided under a license agreement containing restrictions on 
use and disclosure and are protected by intellectual property laws. Except as expres


## Generate BGE-M3 Embeddings

Convert section-based chunks into dense vector representations.

These vectors will be used to build the FAISS index.

In [9]:
embeddings = embedding_model.embed_documents(
    chunk_texts
)

print("=" * 80)

print("EMBEDDING GENERATION")

print("=" * 80)

print(
    f"Embeddings Created: {len(embeddings):,}"
)

EMBEDDING GENERATION
Embeddings Created: 880


## Validate Embedding Dimensions

Verify the number of embeddings created and confirm vector dimensions.

In [10]:
print("=" * 80)

print("EMBEDDING VALIDATION")

print("=" * 80)

print(
    f"Embedding Count: {len(embeddings):,}"
)

print(
    f"Embedding Dimension: {len(embeddings[0])}"
)

EMBEDDING VALIDATION
Embedding Count: 880
Embedding Dimension: 1024


## Prepare Vectors for FAISS

Convert embeddings into NumPy arrays required by FAISS.

In [11]:
import faiss
import numpy as np

In [12]:
vectors = np.array(
    embeddings,
    dtype=np.float32
)

print("=" * 80)

print("VECTOR VALIDATION")

print("=" * 80)

print(
    f"Vector Shape: {vectors.shape}"
)

VECTOR VALIDATION
Vector Shape: (880, 1024)


## Build Native FAISS Index

Create a FAISS IndexFlatL2 store and add all vectors.

In [13]:
dimension = vectors.shape[1]

faiss_index = faiss.IndexFlatL2(
    dimension
)

faiss_index.add(
    vectors
)

print("=" * 80)

print("FAISS INDEX CREATED")

print("=" * 80)

print(
    f"Vectors Indexed: {faiss_index.ntotal:,}"
)

FAISS INDEX CREATED
Vectors Indexed: 880


## Persist FAISS Index

Save the vector index for future retrieval operations.

In [14]:
from pathlib import Path

faiss_dir = Path(
    "../vectorstores/faiss_bgem3"
)

faiss_dir.mkdir(
    parents=True,
    exist_ok=True
)

index_file = (
    faiss_dir /
    "oracle_bgem3.index"
)

faiss.write_index(
    faiss_index,
    str(index_file)
)

print(
    f"FAISS Index Saved: {index_file.resolve()}"
)

FAISS Index Saved: D:\AI\Projects\oracle-rag-comparison-lab\vectorstores\faiss_bgem3\oracle_bgem3.index


## Validate Index Creation

Confirm that all vectors were successfully added.

In [15]:
print("=" * 80)

print("FAISS VALIDATION")

print("=" * 80)

print(
    f"Section Chunks: {len(section_chunks):,}"
)

print(
    f"Indexed Vectors: {faiss_index.ntotal:,}"
)

FAISS VALIDATION
Section Chunks: 880
Indexed Vectors: 880


## Perform Similarity Search

Test retrieval using a sample Oracle ERP question.

In [16]:
query = (
    "How do I create a purchase requisition?"
)

query_embedding = embedding_model.embed_query(
    query
)

In [17]:
query_vector = np.array(
    [query_embedding],
    dtype=np.float32
)

distances, indices = faiss_index.search(
    query_vector,
    k=3
)

print(
    f"Results Returned: {len(indices[0])}"
)

Results Returned: 3


## Review Retrieved Chunks

Inspect the chunks returned by FAISS.

In [18]:
for rank, idx in enumerate(indices[0]):

    print("\n" + "=" * 80)

    print(
        f"RESULT #{rank + 1}"
    )

    print("=" * 80)

    print(
        section_chunks[idx].metadata
    )

    print("\n")

    print(
        section_chunks[idx].page_content[:1000]
    )


RESULT #1
{'source': 'Purchasing_manual.pdf', 'file_type': 'pdf', 'page': 231}


Requisitions    4-7
supplier and internally sourced lines to speed up requisition entry and automatically 
source items to improve buyer efficiency. At any point, the requestor can review the 
requisition online to check the status of the requisition.
Implementing Online Requisitions
Many companies want to provide online requisitions to everyone in the company. 
Others want to limit the number of users who have access to the system. Purchasing is 
flexible enough to meet the needs of your company. To give everyone access to the 
system, simply create a user name for each employee in the company and assign the 
employee the appropriate responsibilities. You can easily designate a smaller group of 
individuals as requisition preparers if you want to limit the number of requestors on the
system. These people can create requisitions for anyone in the company by identifying 
the appropriate requestor directly 

## Export Index Metadata

Save FAISS configuration details for documentation and comparison.

In [19]:
index_info = {
    "Vector Store": "FAISS",
    "Embedding Model": "BGE-M3",
    "Embedding Dimension": 1024,
    "Chunk Count": len(section_chunks),
    "Vector Count": faiss_index.ntotal,
    "Device": device
}

In [20]:
import pandas as pd

index_df = pd.DataFrame(
    index_info.items(),
    columns=[
        "Property",
        "Value"
    ]
)

index_df

,Property,Value
0,Vector Store,FAISS
1,Embedding Model,BGE-M3
2,Embedding Dimension,1024
3,Chunk Count,880
4,Vector Count,880
5,Device,cuda


In [21]:
output_dir = Path(
    "../screenshots/vectorstores"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_file = (
    output_dir /
    "faiss_index_info.csv"
)

index_df.to_csv(
    output_file,
    index=False
)

print(
    f"Saved: {output_file.resolve()}"
)

Saved: D:\AI\Projects\oracle-rag-comparison-lab\screenshots\vectorstores\faiss_index_info.csv


# Conclusion

## FAISS Index

Pipeline 2 now supports:

- Section-Based Chunking
- BGE-M3 Embeddings
- Native FAISS Indexing

## Validation

✅ Embeddings Generated

✅ Vectors Indexed

✅ FAISS Index Persisted

✅ Similarity Search Successful

## Pipeline 2 Status

Documents
→ Section-Based Chunking
→ BGE-M3 Embeddings
→ FAISS
→ Retrieval Ready

## Next Step

Proceed to:

### 09_Retrieval_Comparison.ipynb